In [6]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF, ChiSqSelector, StringIndexer
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [7]:
# Create Spark session
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()


In [9]:
# Read the data
data_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json"
data = spark.read.json(data_path)

In [10]:
# Select relevant columns and drop rows with missing values
data = data.select("reviewText", "category").na.drop()

In [11]:
# Define pipeline stages
indexer = StringIndexer(inputCol="category", outputCol="label")
pattern = r'\s+|\d+|[()\[\]{}.,;:!?\+\-=_\"\'`~#@&*%€$§\\\/]+'
regex_tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern=pattern, gaps=True, minTokenLength=2)
stop_words_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
vectorizer = CountVectorizer(inputCol="filtered_words", outputCol="vectorized_Features")
idf = IDF(inputCol="vectorized_Features", outputCol="features")
chi_selector = ChiSqSelector(numTopFeatures=2000, featuresCol="features", outputCol="selectedFeatures", labelCol="label")

In [13]:
# Creating & fitting pipeline
pipeline = Pipeline(stages=[indexer, regex_tokenizer, stop_words_remover, vectorizer, idf, chi_selector])

fitted_pipeline = pipeline.fit(data)

In [14]:
# Transform data
transformed_data = fitted_pipeline.transform(data)

In [15]:
# Get the ChiSqSelector model and its selected features
chi_selector_model = fitted_pipeline.stages[-1]
selected_indices = chi_selector_model.selectedFeatures

In [16]:
# Get the vocabulary from the vectorizer stage
vocabulary = fitted_pipeline.stages[3].vocabulary

In [17]:
# Get the chi-square values from the ChiSqSelector model
chi_values = chi_selector_model.selectedFeatures

In [18]:
# Create a dictionary for mapping words to their chi-square values
word_chi_dict = {vocabulary[i]: chi_values[i] for i in range(len(selected_indices))}

In [19]:
# Sort the words based on chi-square values
sorted_words = sorted(word_chi_dict.items(), key=lambda x: x[1], reverse=True)

In [22]:
# Writing results to file:
with open("output_ds.txt", "w") as file:
    for word, chi_square in sorted_words:
        file.write(f"{word}: {chi_square}\n")

In [21]:
spark.stop()